In [1]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers faiss-cpu
!pip install -q requests beautifulsoup4
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.9 MB/s eta 0:00:00
✅ All packages installed!


In [2]:
# Cell 2 — Login to HuggingFace Hub
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Logged in!')
except Exception:
    login()

✅ Logged in!


In [3]:
# Cell 3 — Mental health resources data
# Free Canadian + WHO mental health resources

mental_health_data = {
    "sad": [
        "Feeling sad is a normal human emotion. If sadness persists for more than 2 weeks, it may indicate depression. Reach out to someone you trust.",
        "CAMH (Centre for Addiction and Mental Health) Canada offers free resources for depression and sadness. Visit camh.ca or call 1-800-463-2338.",
        "Crisis Services Canada — if you are in crisis call or text 9-8-8. Free, confidential, available 24/7 across Canada.",
        "Physical activity has been proven to reduce sadness. Even a 20-minute walk can improve mood significantly.",
        "Talking to a friend, family member, or counselor about your feelings can help process sadness effectively.",
        "BounceBack Canada — free skill-building program for adults dealing with mild to moderate depression. bouncebackontario.ca",
        "WHO recommends behavioral activation therapy for sadness — engaging in activities you used to enjoy even when motivation is low.",
        "Sleep and nutrition directly affect mood. Poor sleep amplifies sadness. Aim for 7-9 hours consistently.",
        "Mindfulness-based cognitive therapy (MBCT) is proven effective for recurring sadness and depression — many free resources online.",
        "Good2Talk — free professional counseling for post-secondary students in Canada. Call 1-866-925-5454."
    ],
    "angry": [
        "Anger is a natural emotion but chronic anger affects health and relationships. Recognizing triggers is the first step.",
        "CAMH Canada provides anger management resources and referrals. Visit camh.ca for self-help tools.",
        "Deep breathing technique — inhale for 4 counts, hold for 4, exhale for 6. This activates the parasympathetic nervous system.",
        "Taking a 10-minute break before responding when angry prevents regrettable reactions.",
        "Physical exercise is one of the most effective anger management tools — redirects adrenaline productively.",
        "Cognitive restructuring — challenge the thoughts that fuel anger. Ask: is this thought accurate? Is it helpful?",
        "Crisis Services Canada — 9-8-8. If anger is leading to thoughts of harming yourself or others, call immediately.",
        "Distress Centres of Greater Toronto — 416-408-4357. Free counseling for anger and emotional distress.",
        "WHO recommends problem-solving therapy for chronic anger — identify the root cause and address it directly.",
        "Journaling anger can reduce its intensity — writing down what triggered you creates emotional distance."
    ],
    "fear": [
        "Fear and anxiety are the most common mental health challenges. You are not alone — 1 in 4 Canadians experience anxiety disorders.",
        "Anxiety Canada — anxietycanada.com — free evidence-based resources for fear and anxiety management.",
        "Crisis Services Canada — 9-8-8. Available 24/7 for anyone experiencing overwhelming fear or panic.",
        "Grounding technique — 5-4-3-2-1 method: name 5 things you see, 4 you can touch, 3 you hear, 2 you smell, 1 you taste.",
        "CAMH Canada — free anxiety self-assessment and resources. camh.ca/anxiety",
        "Progressive muscle relaxation — tense and release muscle groups systematically to reduce physical fear response.",
        "Cognitive Behavioral Therapy (CBT) is the gold standard for fear and anxiety. Many free CBT workbooks available online.",
        "MindShift CBT app — free Canadian app specifically designed for anxiety management.",
        "Exposure therapy — gradually facing feared situations reduces anxiety over time. Start with the least scary scenario.",
        "WHO recommends regular sleep, exercise, and social connection as foundations for managing chronic fear."
    ],
    "disgust": [
        "Feelings of disgust toward oneself can indicate low self-esteem or trauma. These feelings are treatable.",
        "CAMH Canada offers trauma-informed care resources. camh.ca",
        "Self-compassion practice — treat yourself with the same kindness you would offer a good friend.",
        "Crisis Services Canada — 9-8-8. Free confidential support available 24/7.",
        "Negative self-talk amplifies disgust. Cognitive restructuring helps challenge these thought patterns.",
        "Talk Suicide Canada — 1-833-456-4566. If self-disgust is leading to suicidal thoughts, please call.",
        "Body neutrality — focusing on what your body can do rather than how it looks reduces self-disgust.",
        "Therapy approaches like DBT (Dialectical Behavior Therapy) are specifically effective for intense self-directed emotions.",
        "Kids Help Phone — 1-800-668-6868. For young Canadians struggling with self-image and emotional distress.",
        "WHO emphasizes that feelings of self-disgust are symptoms, not truths. Professional support can help reframe them."
    ],
    "happy": [
        "Great to see you are feeling happy! Positive emotions build resilience for challenging times ahead.",
        "Savoring happiness — take a moment to fully appreciate positive experiences. This strengthens wellbeing.",
        "Sharing happiness with others amplifies it. Reach out to someone you care about today.",
        "Gratitude journaling — writing 3 things you are grateful for daily sustains positive mood.",
        "Use this positive energy to build healthy habits — exercise, sleep, nutrition all become easier when mood is good.",
        "CAMH Canada — mental wellness is not just absence of illness. Actively building positive mental health matters.",
        "Acts of kindness boost happiness for both giver and receiver. Do something kind for someone today.",
        "Mindfulness of positive emotions — notice and fully experience moments of joy without rushing past them.",
        "WHO defines mental health as a state of wellbeing — you are doing well. Keep nurturing it.",
        "Connect with community — social connection is the strongest predictor of long-term happiness and wellbeing."
    ],
    "neutral": [
        "Feeling neutral is completely normal. It is a stable baseline from which you can choose where to direct your energy.",
        "CAMH Canada — mental wellness check-in tools available at camh.ca even when you feel okay.",
        "Use neutral states for reflection — journaling when calm helps process experiences more clearly.",
        "Prevention is better than cure — building mental health habits during neutral periods builds resilience.",
        "Physical activity during neutral moods can shift energy positively without requiring motivation.",
        "Social connection matters even when you feel okay. Reach out to someone you have not spoken to recently.",
        "Crisis Services Canada — 9-8-8. Available even if you are not in crisis — they offer general mental wellness support.",
        "Mindfulness practice is most effective when learned during calm states — harder to learn during crisis.",
        "WHO recommends regular mental health check-ins regardless of current mood.",
        "Sleep hygiene, nutrition, and exercise form the foundation of sustained mental wellbeing."
    ],
    "surprise": [
        "Surprise can be positive or negative. If the surprise was stressful, give yourself time to process.",
        "Unexpected events can trigger anxiety. CAMH Canada resources at camh.ca can help manage adjustment stress.",
        "Crisis Services Canada — 9-8-8. If a surprising event has left you feeling overwhelmed or in crisis.",
        "Grounding techniques help when surprise causes disorientation — focus on your immediate physical surroundings.",
        "Talking through surprising news with a trusted person helps process the emotional impact.",
        "If surprise has caused trauma, CAMH Canada trauma resources can help. camh.ca/trauma",
        "Acceptance — some surprises cannot be undone. Focusing on response rather than the event itself builds resilience.",
        "WHO recommends social support as the most important buffer against negative surprise and unexpected stressors.",
        "Distress Centres of Greater Toronto — 416-408-4357. Free support for processing difficult unexpected events.",
        "Journaling about surprising events helps organize thoughts and feelings into manageable understanding."
    ]
}

total = sum(len(v) for v in mental_health_data.values())
print(f'✅ Mental health data ready!')
print(f'Total resources: {total}')
print(f'Categories: {list(mental_health_data.keys())}')

✅ Mental health data ready!
Total resources: 70
Categories: ['sad', 'angry', 'fear', 'disgust', 'happy', 'neutral', 'surprise']


In [4]:
# Cell 4 — Create embeddings and FAISS indexes per emotion
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

print('Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Separate FAISS index per emotion — more precise search
emotion_indexes = {}
emotion_chunks  = {}

for emotion, resources in mental_health_data.items():
    # Embed all resources for this emotion
    embeddings = embedder.encode(resources, show_progress_bar=False)
    embeddings = np.array(embeddings).astype('float32')

    # Build FAISS index
    dimension = embeddings.shape[1]  # 384
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    emotion_indexes[emotion] = index
    emotion_chunks[emotion]  = resources

    print(f'  {emotion}: {index.ntotal} vectors indexed')

print(f'\n✅ All emotion indexes built!')
print(f'Total indexes: {len(emotion_indexes)}')

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  sad: 10 vectors indexed
  angry: 10 vectors indexed
  fear: 10 vectors indexed
  disgust: 10 vectors indexed
  happy: 10 vectors indexed
  neutral: 10 vectors indexed
  surprise: 10 vectors indexed

✅ All emotion indexes built!
Total indexes: 7


In [5]:
# Cell 5 — Test RAG search
def search_resources(emotion, query, top_k=3):
    query_embedding = embedder.encode([query]).astype('float32')
    index = emotion_indexes[emotion]
    chunks = emotion_chunks[emotion]
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

# Test
print('=== RAG Search Test ===\n')

test_cases = [
    ('sad',     'I feel very sad and hopeless'),
    ('angry',   'I cannot control my anger'),
    ('fear',    'I am very anxious and scared'),
    ('happy',   'I am feeling great today'),
]

for emotion, query in test_cases:
    print(f'Emotion: {emotion}')
    print(f'Query: {query}')
    results = search_resources(emotion, query, top_k=2)
    for i, r in enumerate(results):
        print(f'  Result {i+1}: {r[:100]}...')
    print()

=== RAG Search Test ===

Emotion: sad
Query: I feel very sad and hopeless
  Result 1: Talking to a friend, family member, or counselor about your feelings can help process sadness effect...
  Result 2: Feeling sad is a normal human emotion. If sadness persists for more than 2 weeks, it may indicate de...

Emotion: angry
Query: I cannot control my anger
  Result 1: Physical exercise is one of the most effective anger management tools — redirects adrenaline product...
  Result 2: WHO recommends problem-solving therapy for chronic anger — identify the root cause and address it di...

Emotion: fear
Query: I am very anxious and scared
  Result 1: Exposure therapy — gradually facing feared situations reduces anxiety over time. Start with the leas...
  Result 2: Fear and anxiety are the most common mental health challenges. You are not alone — 1 in 4 Canadians ...

Emotion: happy
Query: I am feeling great today
  Result 1: Great to see you are feeling happy! Positive emotions build resilience

In [6]:
# Cell 6 — Save indexes to Google Drive
from google.colab import drive
import os
import faiss
import pickle

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/emotion_rag_index'
os.makedirs(save_dir, exist_ok=True)

# Save each emotion index separately
for emotion in emotion_indexes:
    faiss.write_index(
        emotion_indexes[emotion],
        f'{save_dir}/faiss_{emotion}.bin'
    )

# Save all chunks
with open(f'{save_dir}/emotion_chunks.pkl', 'wb') as f:
    pickle.dump(emotion_chunks, f)

# Save embedder name for later
with open(f'{save_dir}/config.pkl', 'wb') as f:
    pickle.dump({'embedder': 'all-MiniLM-L6-v2',
                 'emotions': list(emotion_indexes.keys())}, f)

print('✅ All indexes saved to Drive!')
print(f'Files saved:')
for f in os.listdir(save_dir):
    size = os.path.getsize(f'{save_dir}/{f}') / 1024
    print(f'  {f} — {size:.1f} KB')

Mounted at /content/drive
✅ All indexes saved to Drive!
Files saved:
  faiss_sad.bin — 15.0 KB
  faiss_angry.bin — 15.0 KB
  faiss_fear.bin — 15.0 KB
  faiss_disgust.bin — 15.0 KB
  faiss_happy.bin — 15.0 KB
  faiss_neutral.bin — 15.0 KB
  faiss_surprise.bin — 15.0 KB
  emotion_chunks.pkl — 7.5 KB
  config.pkl — 0.1 KB


In [7]:
# Cell 7 — Upload indexes to HuggingFace Hub
from huggingface_hub import HfApi
import os

api = HfApi()
save_dir = '/content/drive/MyDrive/emotion_rag_index'
REPO_ID = 'kashanikram/facial-emotion-vit'

print('Uploading RAG files to HuggingFace Hub...')

# Upload all files
files_to_upload = os.listdir(save_dir)
for filename in files_to_upload:
    filepath = f'{save_dir}/{filename}'
    api.upload_file(
        path_or_fileobj=filepath,
        path_in_repo=filename,
        repo_id=REPO_ID,
        repo_type='model'
    )
    print(f'  ✅ {filename} uploaded!')

print(f'\n✅ All RAG files on HuggingFace Hub!')
print(f'🔗 https://huggingface.co/kashanikram/facial-emotion-vit')
print('\nNext: Notebook 3 — Agent + Gradio Demo!')

Uploading RAG files to HuggingFace Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...n_rag_index/faiss_sad.bin: 100%|##########| 15.4kB / 15.4kB            

  ✅ faiss_sad.bin uploaded!


...

  ✅ emotion_chunks.pkl uploaded!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tion_rag_index/config.pkl: 100%|##########|   121B /   121B            

  ✅ config.pkl uploaded!

✅ All RAG files on HuggingFace Hub!
🔗 https://huggingface.co/kashanikram/facial-emotion-vit

Next: Notebook 3 — Agent + Gradio Demo!
